In [1]:
from pyspark.sql import functions as F

# ============================================
# ÉTAPE 0 : LECTURE DES DONNÉES
# ============================================
storage_account = "energybigdatastorage"
container_raw = "raw"

path_raw = f"abfss://{container_raw}@{storage_account}.dfs.core.windows.net/energy_data_extracted/archive (3).zip/informations_households.csv"

df_hh = spark.read.format("csv") \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .option("sep", ",") \
    .load(path_raw)

print(f"Nombre de lignes : {df_hh.count()}")
print(f"Nombre de colonnes : {len(df_hh.columns)}")
df_hh.printSchema()
df_hh.show(5)

In [2]:
# ============================================
# ÉTAPE 1 : SUPPRIMER LES DOUBLONS
# ============================================
nb_avant = df_hh.count()
df_hh = df_hh.dropDuplicates()
nb_apres = df_hh.count()
print(f"Lignes avant : {nb_avant}")
print(f"Lignes après : {nb_apres}")
print(f"Doublons supprimés : {nb_avant - nb_apres}")

In [3]:
# ============================================
# ÉTAPE 2 : VÉRIFICATION DES VALEURS NULLES
# ============================================
print("=== VALEURS NULLES PAR COLONNE ===")
df_hh.select([F.count(F.when(F.col(c).isNull(), c)).alias(c) for c in df_hh.columns]).show()

# Vérifier aussi les valeurs vides en string
print("=== VALEURS VIDES EN STRING ===")
df_hh.select([F.count(F.when(F.col(c) == "", c)).alias(c) for c in df_hh.columns]).show()

In [4]:
# ============================================
# ÉTAPE 3 : VÉRIFICATION DES VALEURS UNIQUES
# ============================================
print("=== VALEURS UNIQUES PAR COLONNE ===")
for col in df_hh.columns:
    print(f"\n{col} :")
    df_hh.groupBy(col).count().orderBy("count", ascending=False).show()

In [5]:
# ============================================
# ÉTAPE 4 : SAUVEGARDE DANS PROCESSED
# ============================================
container_processed = "processed"

path_processed = f"abfss://{container_processed}@{storage_account}.dfs.core.windows.net/informations_households/"

df_hh.write.format("delta").mode("overwrite").save(path_processed)

print("informations_households sauvegardé dans processed/informations_households/")